

# 🎓 Capstone: The Out-of-Sample Forecasting Challenge

<br>

<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Derivatives-Futures-Chapter-Three/blob/master/colab/Colab_Chapter_Challenge.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Derivatives-Futures-Chapter-Three/master?urlpath=lab/tree/notebooks/Chapter_Challenge.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>

  <a href="https://patrickjhess.github.io/Derivatives-Futures-Chapter-Three/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>

In the previous chapter, *Using Futures to Isolate Sources of Risk and Return*, we explored how to synthetically replicate the risk profile of benchmark equity portfolios. We established a toolkit of four highly liquid futures contracts (ES, NQ, RTY, and YM) and discussed how out-of-sample forecasting is the true test of a model's real-world viability.

We also noted that this framework has **Universal Application**. Once you understand the mechanics of return spanning, you can use these exact same liquid contracts to synthesize a wide variety of custom portfolios.

Now, it is time to put that theory into practice.

### 🎯 The Mission
You want to grab a partner or partners to work with, because in this capstone challenge, you will step into the roles of quantitative researchers. Your objective is to collaborate and use our four-contract futures toolkit to replicate the risk and return profiles of two distinct sector portfolios: **S&P 500 Financials** and **S&P 500 Information Technology**.

### 📚 Learning Objectives
By completing this challenge, you will demonstrate your ability to:
* **Construct a Data Pipeline:** Fetch, join, and clean time-series data for futures contracts, target sector indices, and the SOFR index.
* **Calculate Excess Returns:** Isolate the risk premium of the assets by correctly stripping out an estimated funding rate—SOFR.
* **Model Asset Allocation:** Utilize Ordinary Least Squares (OLS) regression to estimate the optimal weights of your futures basket.
* **Evaluate Real-World Performance:** Generate out-of-sample forecasts and visualize the cumulative tracking error of your synthetic portfolios.

---

### 🏁 The Final Destination
If you're ready to go, get started! Otherwise, five guided steps are suggested below. Each step provides some guidance to help you navigate; however, if you get stuck, every section includes a hidden solution.

Before you get started, click below to see the final destinations.

<details>
<summary><b>📈 The Financial Portfolio Graph</b></summary>
  <img src="https://raw.githubusercontent.com/PatrickJHess/Static_Data_Repo/main/images/SPFIN.png" alt="Financial Portfolio Tracking Error" width="800"/>
</details>

<details>
<summary><b>💻 The Information Technology Portfolio Graph</b></summary>
  <img src="https://raw.githubusercontent.com/PatrickJHess/Static_Data_Repo/main/images/SPINFOTECH.png" alt="Information Technology Portfolio Tracking Error" width="800"/>
</details>

<br>

*Ready to build your models? Run the setup cell below to initialize your environment, and let's get to work!*

### 🛠️ The Starting Point

In order to complete the challenge, you'll need access to the modules and functions from the notebook **🧭 Using Futures to Isolate Sources of Risk and Return**.

Here is a reminder of what you need to set up:

**1. Imports and Instantiations**
* Import `financial_quant` as `fq`
* Import `seaborn` as `sns`
* Import `statsmodels.api` as `sm`
* Instantiate the following:
  * `fred_data` using `fq.FredReader()`
  * `massive_data` using `fq.MASSIVEReader(calls_per_minute=5)`
  * `static_futures_data` from `fq.static_futures_data`

**2. Required Functions**
Ensure the following custom functions defined in the previous notebook are loaded into your environment:
* `futures_data`
* `predict_actual`
* `plot_tracking_error`

*Tip: It is highly recommended that you run the setup and import these functions before you attempt to complete this challenge.*



### 1 &ensp;📥 Fetch and Prepare the Data

#### 1.a&ensp; Fetch the Data
<blockquote>
You will need to pull three different data sources to build your pipeline. Here are the tools and tickers you need for each:

1. **Futures Data:** Use the custom <code>futures_data()</code> function we developed earlier in the notebook to pull the March and June 2026 contracts.
2. **Target Portfolios:** Use the <code>static_futures_data()</code> function. The file names are <code>SPFIN_Index</code> for the Financials sector, and <code>SPTECH_Index</code> for the Information Technology sector. *(Note: Because these are Total Return indices, the resulting columns in your DataFrame will automatically be named <code>SPFINTR_Index</code> and <code>SPTECHTR_Index</code>. Keep an eye on that "TR" for the next steps!)*
3. **SOFR Index:** Use the <code>fred_data.get_series()</code> function to pull the <code>SOFRINDEX</code>.
</blockquote>

<details>
<summary><b>🔍 Solution: Fetching Data</b></summary>

```python
# 1. Futures Data
df_mar6 = futures_data('H6', '2025-12-11', '2026-03-12', clear_logs=True, wait_to_clear=True)
df_jun6 = futures_data('M6', '2026-03-12', '2026-06-11', clear_logs=True, wait_to_clear=True)
df_futures_raw = pd.concat([df_mar6, df_jun6])
df_futures = df_futures_raw.dropna()
display(df_futures.loc['2026-03-12'])

# 2. Target Portfolios
df_spfin = static_futures_data('SPFIN_Index')
df_sptech = static_futures_data('SPTECH_Index')
df_tri = df_spfin.join(df_sptech, how='inner')
df_tri.index = pd.to_datetime(df_tri.index)
display(df_tri.loc['2026-03-12'])

# 3. SOFR Index
sofr_index = fred_data.get_series('SOFRINDEX', start_date=df_futures.index[0], end_date=df_futures.index[-1])
sofr_index.index = pd.to_datetime(sofr_index.index)
display(sofr_index.loc['2026-03-12'])
```
</details>

#### 1.b&ensp; 🧩 Join and Clean The Data
<blockquote>
It's important to remember that the calculation of returns of futures at the roll date requires two futures contracts. A left join is required when joining both the portfolios and the SOFR index (<code>how="left"</code>). Once the DataFrames are joined, clean them up to align dates so returns and excess returns can be calculated accurately.

1. **Join Futures and Target Portfolios:** Use a left join to merge the target portfolios (e.g. <code>df_tri</code>) onto your futures data.
2. **Add SOFR:** Perform another left join to add the SOFR index to this new combined DataFrame.
3. **Clean and Rename:** Drop any missing rows using <code>dropna()</code> to align the dates properly, and rename the <code>SOFRINDEX</code> column to something Python-friendly (like <code>sofr_index</code>).
4. **Safeguard:** Filter out any duplicate columns using <code>~df.columns.duplicated()</code>. *Pro-tip: This prevents errors downstream if you accidentally run this join cell multiple times!*
</blockquote>

<details>
<summary><b>🔍 Solution: Join and Clean</b></summary>

```python
# 1. Left join the portfolios onto the futures data
df_futures_cash = df_futures.join(df_tri, how='left')

# 2. Left join the SOFR index and drop missing data
df_futures_cash_sofr = df_futures_cash.join(sofr_index, how='left').dropna()

# 3. Rename the column for easier access later
df_futures_cash_sofr.rename(columns={'SOFRINDEX': 'sofr_index'}, inplace=True)

# 4. Safeguard: Remove duplicate columns if the cell is run multiple times
df_futures_cash_sofr = df_futures_cash_sofr.loc[:, ~df_futures_cash_sofr.columns.duplicated()]

# Display the roll date to verify
display(df_futures_cash_sofr.loc['2026-03-11':'2026-03-12'])

### 2 &ensp;🧮 Create Excess Return DataFrame

#### 2.a&ensp; Calculate Percentage Changes
<blockquote>
Because the futures contracts overlap, you will actually generate two percentage changes on the roll date. Here is how to handle them:

- **The first instance** is the correct return (calculated purely on the outgoing contract).
- **The second instance** is a bad data point (it calculates a "return" by mixing the price of the old contract with the new contract).

You need to drop that second row. You can easily filter it out by negating the duplicated dates in your index using <code>~df.index.duplicated(keep='first')</code>. Notice how easy Pandas makes it to manipulate time-series data!
</blockquote>

<details>
<summary><b>📊 Solution: Calculating Percentage Changes</b></summary>

```python
# 1. Calculate percentage changes for the entire DataFrame
df_pct_dup = df_futures_cash_sofr.pct_change()

# 2. Filter out the duplicate roll-date row (keep only the first instance)
df_pct_clean = df_pct_dup[~df_pct_dup.index.duplicated(keep='first')]

# Display the roll dates to verify the duplicate is gone
display(df_pct_clean.loc['2026-03-11':'2026-03-12'])

#### 2.b&ensp; ➖💸 Calculate Excess Returns
<blockquote>
Here is your checklist:

1. **Isolate the Futures:** Copy the four futures columns into a new DataFrame (these are already excess returns).
2. **Subtract SOFR:** Use the Pandas <code>sub()</code> method to subtract the <code>sofr_index</code> column from the two sector portfolio columns (<code>SPFINTR_Index</code> and <code>SPTECHTR_Index</code>) along the row axis (<code>axis=0</code>).
3. **Combine and Clean:** Add these newly calculated excess returns to your DataFrame and drop any missing values.
</blockquote>

<details>
<summary><b>📝 Solution: Calculating Excess Returns</b></summary>

```python
# 1. Isolate the Futures (assuming your columns are ES, NQ, RTY, YM)
futures_cols = ['ES', 'NQ', 'RTY', 'YM']
df_excess = df_pct_clean[futures_cols].copy()

# 2. Subtract SOFR from the target portfolios
portfolios = df_pct_clean[['SPFINTR_Index', 'SPTECHTR_Index']]
excess_portfolios = portfolios.sub(df_pct_clean['sofr_index'], axis=0)

# 3. Combine and Clean
df_excess = pd.concat([df_excess, excess_portfolios], axis=1).dropna()

# Display the final excess return dataframe
display(df_excess.head())
```
</details>

### 3.&ensp;⚖️ Use OLS to Estimate Futures Allocations
<blockquote>
When you fit the model, the results object will generate a massive summary. You only need to extract three specific attributes to measure your success:

- **The Allocations:** Use <code>results.params</code> to get the constant and the computed weights for each futures contract.
- **The Standard Errors:** Use <code>results.bse</code> to check the statistical standard error of those allocations.
- **The R-Squared:** Use <code>results.rsquared</code> to measure how accurately your futures basket replicates the target portfolio.

*Run this regression process for both the Financials and Information Technology portfolios to compare their weights.*
</blockquote>

<details>
<summary><b>📐 Solution: OLS Allocation Estimates</b></summary>

```python
import statsmodels.api as sm
import numpy as np

# 1. Independent (X) variables for both portfolios
futures = df_excess_returns[['SP500_FUT', 'RUSS_FUT', 'NSDQ_FUT', 'DJIA_FUT']].dropna()

# 2. Add a constant to the futures data
X = sm.add_constant(futures).rename(columns={'const': 'Delta Funding'})

# --- Financials OLS ---
y_fin = df_excess_returns['SPFINTR_Index'].dropna()
results_fin = sm.OLS(y_fin, X).fit()

df_res_fin = pd.DataFrame([results_fin.params, results_fin.bse], index=['Allocation', 'Std. Error'])
df_res_fin['R-squared'] = [results_fin.rsquared, np.nan]
print("Financials Allocations:")
display(df_res_fin)

# --- Information Technology OLS ---
y_tech = df_excess_returns['SPTECHTR_Index'].dropna()
results_tech = sm.OLS(y_tech, X).fit()

df_res_tech = pd.DataFrame([results_tech.params, results_tech.bse], index=['Allocation', 'Std. Error'])
df_res_tech['R-squared'] = [results_tech.rsquared, np.nan]
print("Info Tech Allocations:")
display(df_res_tech)
```

</details>

### 4.&ensp;🛤️ Calculate the Cumulative Tracking Error
<blockquote>
First, you need to set your training window. Calculate the number of observations (<code>NOBS</code>) by slicing your actual excess returns up to and including the roll date of March 12, 2026. This ensures the model initially trains on data outside of your forecast period. Keep <code>dropna()</code> so you don't accidentally include missing data.

Here is your checklist:

- **Calculate NOBS:** Slice your target portfolio data up to <code>'2026-03-12'</code> and get the length using <code>len()</code>.
- **Generate Forecasts:** Pass your target portfolio and futures data into <code>predict_actual()</code> to get the forecasted and actual returns.
- **Calculate Cumulative Error:** Subtract the forecast from the actual returns, add 1, and use Pandas <code>cumprod()</code> to construct an index representing your tracking error over time.
- **Repeat:** Do this for both the Financials and Information Technology portfolios.
</blockquote>

<details>
<summary><b>✖️ Solution: Calculate Cumulative Product</b></summary>

```python
# 1. Clean the entire dataset at once so X and y align perfectly
df_clean = df_excess_returns.dropna()

# 2. Define the futures toolkit (X) once for both models
df_x = df_clean[['SP500_FUT','RUSS_FUT','NSDQ_FUT','DJIA_FUT']]

# --- Financial Sector Portfolio ---
df_y_fin = df_clean['SPFINTR_Index']

# Train up to and including the roll date
NOBS_fin = len(df_y_fin.loc[:'2026-03-12']) + 1

# Get forecast and actuals
forecast_fin, actual_fin = predict_actual(df_y_fin, df_x, NOBS=NOBS_fin)

# Calculate cumulative tracking error index
financial_error = (1 + pd.Series(actual_fin - forecast_fin, index=df_y_fin.index)).cumprod()


# --- Information Technology Sector Portfolio ---
df_y_tech = df_clean['SPTECHTR_Index']

# Train up to and including the roll date
NOBS_tech = len(df_y_tech.loc[:'2026-03-12']) + 1

# Get forecast and actuals
forecast_tech, actual_tech = predict_actual(df_y_tech, df_x, NOBS=NOBS_tech)

# Calculate cumulative tracking error index
infotech_error = (1 + pd.Series(actual_tech - forecast_tech, index=df_y_tech.index)).cumprod()
```
</details>

### 5.&ensp;📊 Ready To Graph
<blockquote>
You have made it to the finish line! All the heavy lifting is done.

Now, just pass your cumulative tracking error data into the custom <code>plot_tracking_error</code> function provided earlier in the notebook. Be sure to label them clearly using the <code>portfolio_name</code> argument.

If your data pipeline and forecasting models worked correctly, you ought to get the exact same charts shown at the very top of this challenge. Great work!
</blockquote>

<details>
<summary><b>📉 Solution: Plot Tracking Error</b></summary>

```python
# Plot the Financials tracking error
plot_tracking_error(financial_error, portfolio_name='SP Financials')

# Plot the Information Technology tracking error
plot_tracking_error(infotech_error, portfolio_name='SP Information Tech')
```
</details>